<div dir=ltr align=center>
<img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200" class="saturate" >

<br>
<font face="Times New Roman">
<div dir=ltr align=center>
<font color=0F5298 size=7>
 Deep Learning <br>
<font color=2565AE size=5>
Computer Engineering Department - Spring 2025  <br>
<font color=3C99D size=5>
          Homework 3: Practical - Oil Price Forecasting
 <br>
<font color=696880 size=4>
            Designer: Mohammad Amanlou
    
    

# 🛢️ Oil Price Prediction using Time Series Models 📈

This notebook is designed for students to complete tasks related to oil price prediction using different machine learning models. 🚀

## 📚 References
- 📊 [Dataset: Yahoo Finance - CL=F](https://finance.yahoo.com/quote/CL=F/history/)
- 📄 [Reference Paper](https://www.ijournalse.org/index.php/ESJ/article/view/21497)

## 1️⃣ Introduction
🔍 One of the most common applications of recurrent neural networks is **time series forecasting**. In this assignment, you will predict **crude oil prices** using four different methods. 💡

## 2️⃣ Dataset and Preprocessing (25 Points)

### 📥 2.1 Download Dataset
📌 Download the dataset from **Yahoo Finance** for `CL=F` from **2010 to the present**.
[Yahoo Finance - CL=F](https://finance.yahoo.com/quote/CL=F/history/)

### 🎯 2.2 Select Features
✅ Select the `Adj Close` column as the **main feature**.

### ⚠️ **2.3 Handle Missing Data**

You will encounter missing data (`null` values) within your dataset. Follow these detailed steps carefully to handle the missing values and create a complete, reliable dataset:

#### 📝 Step 1: Introduce Random Missing Data
- Identify all indices in the `Adj Close` column that currently have valid (non-null) data.
- Set a random seed (`np.random.seed(42)`) for reproducibility.
- Randomly select **10%** of these valid indices and set their values to `NaN`.

#### 🔍 Step 2: Identify Missing Values
- Identify all dates where at least one column has a missing value (`NaN`).
- Print the number of missing dates and the total number of dates to evaluate the extent of missingness.

#### 🔧 Step 3: Replace Missing Values
- Create a copy of the `Adj Close` column for filling purposes.
- First, apply **linear interpolation** to estimate missing values based on surrounding data points.
- Then, use backward fill (`bfill`) followed by forward fill (`ffill`) methods to handle any remaining missing values at the start or end of the dataset.

#### 🎯 Outcome:
After completing these steps, your dataset will have no missing values in the `Adj Close` column, ready for further analysis or modeling.

🛠 *Your task:* Implement the missing data handling methods below. (16 Points)

### catching the data

In [ ]:
# pip install yfinance

In [ ]:
import yfinance as yf

data = yf.download(
    "CL=F",
    start="2010-08-05",
    end="2026-08-04"
)

print(data.head())

### defining the root directories

In [ ]:
data_root = "E:/Study/Deep Learning/4 - Exercises/0 - Source books/3 - Dr. Soleymani/Myself/3 - HW3/Practical/data/Q1_RNN/"
Results_root = "E:/Study/Deep Learning/4 - Exercises/0 - Source books/3 - Dr. Soleymani/Myself/3 - HW3/Practical/Results/Q1_RNN/"
Runs = "E:/Study/Deep Learning/4 - Exercises/0 - Source books/3 - Dr. Soleymani/Myself/3 - HW3/Practical/Runs/Q1_RNN/"

In [ ]:
import pandas as pd
data_pd = pd.DataFrame(data)
print('saving csv type')
data_pd.to_csv(data_root +'oil price 2010 to 2026.csv')

### exploring the data

In [ ]:
data_pd

In [ ]:
data_pd.to_excel(data_root + 'oil price 2010 to 2026.xlsx')

In [ ]:
print(data_pd.index.min())
print(data_pd.index.max())
print(len(data_pd))
print(data_pd.index.duplicated().sum())
print(data_pd.isna().sum())

In [ ]:
# TO DO: Identify missing dates and null values

calendar_days = (data.index.max() - data.index.min()).days + 1
trading_days = len(data)

print("Calendar days:", calendar_days)
print("Trading rows:", trading_days)
print(f"number of missing rows: \n{calendar_days - trading_days}" )

### filling up the calender with NaN numbers for missing dates

In [ ]:
data.isna().sum()

In [ ]:
data.isnull().sum()

In [ ]:
# TO DO: Introduce random null
import numpy as np
random_state = np.random.seed(42)

full_dates = pd.date_range(
    start=data.index.min(),
    end=data.index.max(),
    freq="D"      # Daily
)

In [ ]:
full_dates

In [ ]:
full_data = data.reindex(full_dates)

In [ ]:
# TO DO: Identify missing dates and null values
print(f"Num of missing values: \n{full_data.isnull().sum()}\n")
full_data

In [ ]:
# TO DO: Fill missing values using .interpolate or .fillna(method='bfill').fillna(method='ffill')  

full_data_filled = full_data.ffill()
print(f"Num of missing values: \n{full_data_filled.isnull().sum()}\n")
full_data_filled

#### saving the filled dataset

In [ ]:
full_data_filled.to_csv(Results_root + 'full_data_filled.csv')
full_data_filled.to_excel(Results_root + 'full_data_filled.xlsx')

### ✂️ 2.4 Train-Test Split and Normalization
- **Split** the dataset into **training and test sets** based on the ratio given in the reference paper.
- **Normalize** the data.

📄 [Reference Paper](https://www.ijournalse.org/index.php/ESJ/article/view/21497)

🛠 *Your task:* Implement the splitting and normalization below. (4 Points)

#### Well, first of all, it should be said that this link is not available. At least I tried it with 10 different regions in VPN and it did not open. But in general, I don't think this division will cause any problems in the project process if we do it with standard settings.

#### There is an ambiguity here. In compiling the notebook by the designers, only one column is normalized. If we look at the dataset, we see that it has different columns, we first normalize every column, and then in the future, if the designers say so or we see references in the code that contradict this, we change the method.

In [ ]:
cols = [col[0] for col in full_data_filled.columns]

print(cols)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

def temporal_split(series, train_ratio=0.6, val_ratio=0.2):
    #TO DO
    train_set_ratio = int(len(series) * train_ratio)
    val_set_ratio = int(len(series) * val_ratio)
    test_set_ratio = int(len(series) * (1 - (train_ratio + val_ratio)))

    print(f"{train_set_ratio}, {val_set_ratio}, {test_set_ratio}")

    train_val, test = train_test_split(series, test_size= test_set_ratio, 
                                       shuffle=False)   
    train , val = train_test_split(train_val, test_size=val_set_ratio , shuffle=False)

    return train, val, test

train_list, val_list, test_list = [], [], []
filled_data = full_data_filled
train, val, test = temporal_split(filled_data)
train_list.append(train)
val_list.append(val)
test_list.append(test)

train_data = pd.concat(train_list)
val_data = pd.concat(val_list)
test_data = pd.concat(test_list)


scaler = MinMaxScaler()
for col in cols:
    train_data[col] = scaler.fit_transform(train_data[col])
    val_data[col] = scaler.transform(val_data[col])
    test_data[col] = scaler.transform(test_data[col])

print("Training data sample:")
print(train_data.head())
print("Testing data sample:")
print(test_data.head())

In [ ]:
print("Training data sample:")
print(train_data["Close"].head())
print("Testing data sample:")
print(test_data["Close"].head())

#### saving

In [ ]:
train_data.to_csv(Results_root + "train_data.csv")
train_data.to_excel(Results_root + "train_data.xlsx")

val_data.to_csv(Results_root + "val_data.csv")
val_data.to_excel(Results_root + "val_data.xlsx")

test_data.to_csv(Results_root + "test_data.csv")
test_data.to_excel(Results_root + "test_data.xlsx")

### 📊 2.5 Data Visualization
- **Plot a histogram** similar to **Figure 6** in the reference paper, showing the **distribution of oil prices**.

🛠 *Your task:* Implement the histogram plot below. (5 Points)

In [ ]:
# TO DO: Plot histogram of 'Adj Close'

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

train_data["Close"].plot.hist(
    bins=20,
    color = 'grey',
    edgecolor="black",
    legend = False
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Histogram of 'Close'")
plt.savefig(Results_root + 'Histogram of Close')
plt.show()

## 3️⃣ Implementing Deep Learning Models 🤖 (60 Points)

The reference paper utilizes **three models** for time series forecasting:
- `RNN`
- `LSTM`
- `GRU`

📌 **Train** each model using the **hyperparameters** given in **Table 4** of the paper.
📌 Use `Mean Square Error (MSE)` as the **loss function**.

📄 [Reference Paper](https://www.ijournalse.org/index.php/ESJ/article/view/21497)


### Important Details & Clarifications

- **What to Predict?**  
  The goal is to predict **the actual next-day price** (regression problem), rather than just identifying price increase or decrease.
  
- **Input/Output Structure:**  
  - **Input:** A window of \( k \) consecutive daily prices (e.g., 50 days).  
  - **Output:** The predicted price for the next day.
  
- **How to Evaluate?**  
  Use the four metrics (RMSE, MAE, MAPE, \( R^2 \)) to gauge how accurately your model tracks the real price values.

- **Target Accuracy:**  
  Your accuracy might differ from the paper’s due to factors like data splitting, normalization, or different random seeds. However, aim to closely replicate the paper’s results or provide justifications for any discrepancy.

**Final Deliverables:**
1. **All four trained models** (RNN, LSTM, GRU).  
2. **Comparison plots** of predicted vs. actual values (in both normalized and original price scales, if desired).  
3. **Performance metrics** (RMSE, MAE, MAPE, \( R^2 \)) for each model, presented in a table or a concise summary.


🛠 *Your task:* Implement these models below. (30 Points)

#### Given that the referenced article is not available, we will proceed with these hyperparameters for now:

<div align='center'>

| Hyperparameter     | RNN     | LSTM    | GRU     |
| :------------------: | :-------: | :-------: | :-------: |
| Window Size        | 50      | 50      | 50      |
| Prediction Horizon | 1       | 1       | 1       |
| Hidden Units       | 64      | 64      | 64      |
| Number of Layers   | 2       | 2       | 2       |
| Dropout            | 0.2     | 0.2     | 0.2     |
| Batch Size         | 32      | 32      | 32      |
| Learning Rate      | 0.001   | 0.001   | 0.001   |
| Optimizer          | Adam    | Adam    | Adam    |
| Loss               | RMSE, MAE, MAPE, R^2 | RMSE, MAE, MAPE, R^2 | RMSE, MAE, MAPE, R^2 |
| Epoch              | 50     | 50     | 50     |
| Shuffle            | False   | False   | False   |
| Random Seed        | 42      | 42      | 42      |

</div>

In [ ]:
import torch

device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def create_sequences(data, window_size, target_col="Close"):
    """
    a function to split data to window sizes for each time step
    and a target value to predict    
    """
    feature_values = data.to_numpy(dtype=np.float32)
    target_values = data[target_col].to_numpy(dtype=np.float32)

    X, y = [], []

    for i in range(len(data) - window_size):
        X.append(feature_values[i: i + window_size])
        y.append(target_values[i + window_size])

    X = torch.tensor(np.array(X), dtype=torch.float32)
    y = torch.tensor(np.array(y), dtype=torch.float32)
    print(y.shape)

    return X, y


window_size = 50
batch_size = 32

X_train, y_train = create_sequences(train_data, window_size)
X_val, y_val = create_sequences(val_data, window_size)
X_test, y_test = create_sequences(test_data, window_size)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


#### saving

In [ ]:
torch.save({
    "X_train": X_train,
    "y_train": y_train,

    "X_val": X_val,
    "y_val": y_val,

    "X_test": X_test,
    "y_test": y_test,

    "window_size": window_size,
    "batch_size": batch_size,
    "feature_names": cols,
}, Results_root + "oil_dataset.pt")


In [ ]:
# LSTM
class LSTMModel(nn.Module):
    def __init__(
        self,
        input_size=len(train_data.columns),
        hidden_size=64, # don't confuse with window_size
        num_layers=2,
        output_size=1,
        dropout=0.2
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(
            in_features=hidden_size,
            out_features=output_size
        )

    def forward(self, x):
        # lstm_out shape:
        # (batch_size, sequence_length, hidden_size)
        lstm_out, _ = self.lstm(x)

        # output of the last timestep
        last_output = lstm_out[:, -1, :]

        # hidden cells (64) :-> FC network :-> 1 neuron
        prediction = self.fc(last_output)

        return prediction

lstm_model = LSTMModel().to(device)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs=50, device=None):
    if device is None:
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
    else:
        device = torch.device(device)
        model = model.to(device)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # -------------------------
        # Training
        # -------------------------
        model.train()

        total_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = loss_fn(predictions, y_batch)

            loss.backward()

            optimizer.step()

            total_train_loss += loss.item() * X_batch.size(0)

        average_train_loss = (
            total_train_loss / len(train_loader.dataset)
        )

        train_losses.append(average_train_loss)

        # -------------------------
        # Validation
        # -------------------------
        model.eval()

        total_val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)

                loss = loss_fn(predictions, y_batch)

                total_val_loss += loss.item() * X_batch.size(0)

        average_val_loss = (
            total_val_loss / len(val_loader.dataset)
        )

        val_losses.append(average_val_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch [{epoch + 1:02d}/{epochs}] | "
                f"Train Loss: {average_train_loss:.6f} | "
                f"Validation Loss: {average_val_loss:.6f}"
            )

    return train_losses, val_losses

train_losses, val_losses = train_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=50
)


#### saving the LSTM model

In [ ]:
checkpoint = {
    # Model
    "model_state_dict": lstm_model.state_dict(),

    # Optimizer
    "optimizer_state_dict": optimizer.state_dict(),

    # Training information
    "epochs": 50,
    "train_losses": train_losses,
    "val_losses": val_losses,

    # Model hyperparameters
    "model_config": {
        "input_size": X_train.shape[2],
        "hidden_size": 64,
        "num_layers": 2,
        "output_size": 1,
        "dropout": 0.2,
    },

    # Data configuration
    "data_config": {
        "window_size": window_size,
        "batch_size": batch_size,
        "feature_names": cols,
        "target_col": "Close",
    },
}

torch.save(checkpoint, Results_root + "LSTM_oil_checkpoint.pt")

In [ ]:
lstm_model.parameters

In [ ]:
class RNNModel(nn.Module):
    #TO DO
    def __init__(
    self,
    input_size=len(train_data.columns),
    hidden_size=64, # don't confuse with window_size
    num_layers=2,
    output_size=1,
    dropout=0.2
):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(
            in_features=hidden_size,
            out_features=output_size
        )

    def forward(self, x):
        # rnn_out shape:
        # (batch_size, sequence_length, hidden_size)
        rnn_out, _ = self.rnn(x)

        # output of the last timestep (-1 in the next line indicates the last outputs of the series)  
        last_output = rnn_out[:, -1, :] 

        # hidden cells (64) :-> FC network :-> 1 neuron
        prediction = self.fc(last_output)

        return prediction

rnn_model = RNNModel().to(device)
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

train_losses, val_losses = train_model(
    model=rnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=50
)

In [ ]:
checkpoint = {
    # Model
    "model_state_dict": rnn_model.state_dict(),

    # Optimizer
    "optimizer_state_dict": optimizer.state_dict(),

    # Training information
    "epochs": 50,
    "train_losses": train_losses,
    "val_losses": val_losses,

    # Model hyperparameters
    "model_config": {
        "input_size": X_train.shape[2],
        "hidden_size": 64,
        "num_layers": 2,
        "output_size": 1,
        "dropout": 0.2,
    },

    # Data configuration
    "data_config": {
        "window_size": window_size,
        "batch_size": batch_size,
        "feature_names": cols,
        "target_col": "Close",
    },
}

torch.save(checkpoint, Results_root + "RNN_oil_checkpoint.pt")

In [ ]:
rnn_model.parameters

In [ ]:
# GRU
class GRUModel(nn.Module):
    #TO DO
    def __init__(
    self,
    input_size=len(train_data.columns),
    hidden_size=64, # don't confuse with window_size
    num_layers=2,
    output_size=1,
    dropout=0.2
):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(
            in_features=hidden_size,
            out_features=output_size
        )

    def forward(self, x):
        # gru_out shape:
        # (batch_size, sequence_length, hidden_size)
        gru_out, _ = self.gru(x)

        # output of the last timestep (-1 in the next line indicates the last outputs of the series)  
        last_output = gru_out[:, -1, :] 

        # hidden cells (64) :-> FC network :-> 1 neuron
        prediction = self.fc(last_output)

        return prediction

gru_model = GRUModel().to(device)
optimizer = torch.optim.Adam(gru_model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

train_losses, val_losses = train_model(
    model=gru_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=50
)

In [ ]:
checkpoint = {
    # Model
    "model_state_dict": gru_model.state_dict(),

    # Optimizer
    "optimizer_state_dict": optimizer.state_dict(),

    # Training information
    "epochs": 50,
    "train_losses": train_losses,
    "val_losses": val_losses,

    # Model hyperparameters
    "model_config": {
        "input_size": X_train.shape[2],
        "hidden_size": 64,
        "num_layers": 2,
        "output_size": 1,
        "dropout": 0.2,
    },

    # Data configuration
    "data_config": {
        "window_size": window_size,
        "batch_size": batch_size,
        "feature_names": cols,
        "target_col": "Close",
    },
}

torch.save(checkpoint, Results_root + "GRU_oil_checkpoint.pt")

### 📈 3.1 Prediction and Evaluation
1. **Prediction:** After training, generate predictions for the test set (i.e., predict the next-day price based on the preceding \( k \) days).
2. **Visualization:** **Plot the predicted values** alongside the **actual values** for each model. This comparison helps in visually assessing each model’s performance.

🛠 **Your Task:** Implement the **visualization of predictions** (15 Points).

In [ ]:
print(lstm_model.state_dict)
print(rnn_model.state_dict)
print(gru_model.state_dict)

In [ ]:
#TO DO: predict real outputs
# Predictions
def predict(model, data_loader):
    model.eval()

    predictions = []
    real_outputs = []
    model.to(device)

    with torch.no_grad():

        for X_batch, y_batch in data_loader:
            
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            prediction = model(X_batch)

            predictions.append(prediction.cpu())
            real_outputs.append(y_batch.cpu())

    predictions = torch.cat(predictions, dim=0)
    real_outputs = torch.cat(real_outputs, dim=0)

    return predictions, real_outputs

In [ ]:
test_predictions, test_real = predict(
    lstm_model,
    test_loader
)

In [ ]:
import matplotlib.pyplot as plt

def plot_predictions(predictions, actual, model_name):
    
    # Tensor -> NumPy
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()

    if torch.is_tensor(actual):
        actual = actual.detach().cpu().numpy()

    # Flatten (N, 1) -> (N,)
    predictions = predictions.flatten()
    actual = actual.flatten()


    plt.figure(figsize=(12, 5))

    plt.plot(actual, label="Actual Values")
    plt.plot(predictions, label=f"{model_name} Predictions")

    plt.title(f"{model_name} Predictions vs Actual Values")
    plt.xlabel("Time Step")
    plt.ylabel("Normalized Price")
    plt.legend()
    plt.grid(True)
    plt.savefig(Results_root + f"{model_name} Predictions vs Actual Values")
    plt.show()

In [ ]:
our_models_names = ["LSTM", "RNN", "GRU"]
models = [lstm_model, rnn_model, gru_model]

for model_name, model in zip(our_models_names, models):

    test_predictions, test_real = predict(
        model,
        test_loader
    )

    plot_predictions(
        test_predictions,
        test_real,
        model_name
    )

### 📊 3.2 Error Metrics
📌 Explain the following **error metrics** used in the paper:
- `RMSE` (Root Mean Square Error)
- `MAE` (Mean Absolute Error)
- `MAPE` (Mean Absolute Percentage Error)
- `R-Squared` (Coefficient of Determination)

**📌 Instruction:**  
Explain each of these error metrics and calculate them for **each model** (RNN, LSTM, GRU). Compare your results with the paper’s findings to see how closely they match.

🛠 *Your task:* Implement the evaluation metrics below. (15 Points)

In [ ]:
# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_metrics(predictions, actual):
    # Tensor -> NumPy
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()

    if torch.is_tensor(actual):
        actual = actual.detach().cpu().numpy()

    # (N, 1) -> (N,)
    predictions = np.array(predictions).flatten()
    actual = np.array(actual).flatten()

    # RMSE
    rmse = np.sqrt(
        mean_squared_error(actual, predictions)
    )

    # MAE
    mae = mean_absolute_error(actual, predictions)

    # MAPE
    mape = np.mean(
        np.abs((actual - predictions) / actual)
    ) * 100

    # R²
    r2 = r2_score(actual, predictions)

    return {
        "RMSE": round(float(rmse), 4),
        "MAE": round(float(mae), 4),
        "MAPE": round(float(mape), 4),
        "R2": round(float(r2), 4)
    }

rnn_predictions, actual_values = predict(rnn_model, test_loader)
lstm_predictions, _ = predict(lstm_model, test_loader)
gru_predictions, _ = predict(gru_model, test_loader)

print("RNN Metrics (Unscaled):", calculate_metrics(rnn_predictions, actual_values))
print("LSTM Metrics (Unscaled):", calculate_metrics(lstm_predictions, actual_values))
print("GRU Metrics (Unscaled):", calculate_metrics(gru_predictions, actual_values))

In [ ]:
full_data_filled.columns

In [ ]:
# Fill missing values for all features
filled_data = data.copy()
for column in ['Open', 'High', 'Low', 'Volume', 'Close']:
    filled_data[column] = filled_data[column].interpolate(method='linear').bfill().ffill()

# WE WILL USE THESE IN ARIMA PART AS INPUTS OF MODELS
train_data = filled_data.iloc[:-int(0.3 * len(filled_data))]        # 70% of the data
test_data = filled_data.iloc[-int(0.3 * len(filled_data)):]     # 30% of the data
train_target = train_data['Close']
train_exog = train_data[['Open', 'High', 'Low', 'Volume']]
test_exog = test_data[['Open', 'High', 'Low', 'Volume']]

# ADF Test
from statsmodels.tsa.stattools import adfuller
result = adfuller(train_data['Close'])
critical_values = {
    key: round(float(value), 3) for key, value in result[4].items()
}
print(f"ADF Statistic: {result[0]:.3f}")
print(f"p-value: {result[1]:.3f}")
print(f"Critical Values: {critical_values}")
print("The series is stationary." if result[1] < 0.05 else "The series is not stationary.")

## 4️⃣ ARIMA Model 📉 (15 Points)

📌 Explain the **difference** between `ARIMA` and `SARIMA` models.

📌 List the **advantages** and **limitations** of `ARIMA`.

📌 Explain the **mathematical formulation** of `ARIMA`, including its **parameters**.

📌 Determine the **optimal parameters** for `ARIMA` and **report the results**.

📌 Compare the results with **Table 6** from the paper.

📄 [Reference Paper](https://www.ijournalse.org/index.php/ESJ/article/view/21497)

🛠 *Your task:* Implement the ARIMA model below.

# ARIMA Model

## 1. Difference Between ARIMA and SARIMA

**ARIMA (AutoRegressive Integrated Moving Average)** is mainly used for modeling non-seasonal time-series data.

The model is represented as:

$$
ARIMA(p,d,q)
$$

where:

- $p$: Number of autoregressive (AR) terms.
- $d$: Number of differencing operations required to achieve stationarity.
- $q$: Number of moving-average (MA) terms.

**SARIMA (Seasonal AutoRegressive Integrated Moving Average)** extends ARIMA by adding explicit seasonal components:

$$
SARIMA(p,d,q)(P,D,Q)_s
$$

where:

- $P$: Seasonal autoregressive order.
- $D$: Seasonal differencing order.
- $Q$: Seasonal moving-average order.
- $s$: Length of the seasonal cycle.

Therefore, **ARIMA models non-seasonal temporal dependencies**, while **SARIMA can additionally capture repeating seasonal patterns**.

---

## 2. Advantages and Limitations of ARIMA

### Advantages

- Provides a well-established statistical framework for time-series forecasting.
- Models both autoregressive and moving-average dependencies.
- Can handle many non-stationary series through differencing.
- Relatively interpretable compared with deep-learning models.
- Can perform well when temporal relationships are approximately linear.
- Provides a useful statistical baseline for comparison with RNN, LSTM, and GRU.

### Limitations

- Primarily captures linear relationships.
- Usually requires the series to become stationary after differencing.
- Standard ARIMA does not explicitly model seasonality.
- Performance depends on appropriate selection of $p$, $d$, and $q$.
- May struggle when the statistical behavior of the series changes significantly over time.
- Can be sensitive to extreme events and structural changes.

---

# 3. Mathematical Formulation of ARIMA

ARIMA combines three main components:

1. **AR — AutoRegressive**
2. **I — Integrated**
3. **MA — Moving Average**

Therefore:

$$
ARIMA(p,d,q)
$$

---

## 3.1 AutoRegressive Component — AR(p)

The autoregressive component uses previous observations of the time series to estimate the current value.

$$
y_t =
c +
\phi_1 y_{t-1}
+
\phi_2 y_{t-2}
+
\cdots
+
\phi_p y_{t-p}
+
\epsilon_t
$$

or equivalently:

$$
y_t =
c +
\sum_{i=1}^{p} \phi_i y_{t-i}
+
\epsilon_t
$$

where:

- $y_t$: Current value.
- $y_{t-i}$: Previous observations.
- $\phi_i$: Autoregressive coefficients.
- $p$: Number of lagged observations.
- $c$: Constant.
- $\epsilon_t$: Random error.

For example, an AR(2) model is:

$$
y_t =
c +
\phi_1y_{t-1}
+
\phi_2y_{t-2}
+
\epsilon_t
$$

---

## 3.2 Integrated Component — I(d)

The Integrated component applies **differencing** to make a non-stationary time series stationary.

For first-order differencing:

$$
\Delta y_t = y_t-y_{t-1}
$$

Therefore, if one differencing operation is sufficient:

$$
d=1
$$

If the original series is already stationary:

$$
d=0
$$

For second-order differencing:

$$
\Delta^2y_t
=
\Delta y_t-\Delta y_{t-1}
$$

which corresponds to:

$$
d=2
$$

The **Augmented Dickey-Fuller (ADF) test** can be used to evaluate whether sufficient evidence exists to consider the series stationary.

---

## 3.3 Moving Average Component — MA(q)

The Moving Average component uses previous **forecast errors**.

$$
y_t =
c +
\epsilon_t +
\theta_1\epsilon_{t-1}
+
\theta_2\epsilon_{t-2}
+
\cdots +
\theta_q\epsilon_{t-q}
$$

or:

$$
y_t =
c +
\epsilon_t +
\sum_{j=1}^{q}\theta_j\epsilon_{t-j}
$$

where:

- $\epsilon_t$: Current error.
- $\epsilon_{t-j}$: Previous forecast errors.
- $\theta_j$: Moving-average coefficients.
- $q$: Number of lagged error terms.

---

# 4. Complete ARIMA Formulation

After differencing the original series $d$ times, define the stationary series as:

$$
y'_t = \Delta^d y_t
$$

The ARIMA model can then be written as:

$$
y'_t =
c
+
\sum_{i=1}^{p}\phi_i y'_{t-i}
+
\epsilon_t
+
\sum_{j=1}^{q}\theta_j\epsilon_{t-j}
$$

Therefore:

$$
\boxed{ARIMA(p,d,q)}
$$

where:
<div align= center>

| Parameter | Component | Meaning |
|---|---|---|
| $p$ | AR | Number of lagged observations |
| $d$ | Integrated | Number of differencing operations |
| $q$ | MA | Number of lagged forecast errors |

</div>
---

# 5. Determining the Optimal ARIMA Parameters

The optimal values of $p$, $d$, and $q$ can be determined using `auto_arima`.

```python
from pmdarima import auto_arima

arima_model = auto_arima(
    train_target,
    seasonal=False,
    stepwise=True,
    trace=True,
    suppress_warnings=True
)

print(f"Optimal ARIMA Order: {arima_model.order}")

In [ ]:
# Train ARIMA model using auto_arima
#pip install pmdarima

In [ ]:
# Train ARIMA model using auto_arima
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

#TO DO:Find optimal arima model using auto_arima

arima_model = auto_arima(
    train_target,
    seasonal=False,
    stepwise=True,
    trace=True,
    suppress_warnings=True
)

print(f"Optimal ARIMA Order: {arima_model.order}")

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from tqdm import tqdm
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=ConvergenceWarning, module="statsmodels")

#TO DO: Predinct Arima outputs
# ARIMA Predictions

history = list(train_target.values)

arima_predictions = []

for i in tqdm(range(len(test_data))):

    model = ARIMA(
        history,
        order=arima_model.order
    )

    model_fit = model.fit()

    prediction = model_fit.forecast(steps=1)[0]

    arima_predictions.append(prediction)

    # add the real observed value to history
    history.append(test_data["Close"].iloc[i])


arima_predictions = pd.Series(
    arima_predictions,
    index=test_data.index
)

print(arima_predictions)

In [ ]:
test_real = test_data["Close"].to_numpy()
test_real.shape

In [ ]:
arima_predictions_np = arima_predictions.to_numpy()
arima_predictions_np.shape

In [ ]:
plot_predictions(
    arima_predictions_np,
    test_real,
    "ARIMA"
)

In [ ]:
train_target = train_target.squeeze()

In [ ]:
print(type(train_target))
print(train_target.shape)

In [ ]:
test_real = test_data.squeeze()

In [ ]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX


order_combinations = list(
    itertools.product(
        range(0,3),
        [1],
        range(0,3)
    )
)

seasonal_combinations = list(
    itertools.product(
        range(0,2),
        [0,1],
        range(0,2),
        [7]
    )
)

# Initialize variables to store the best results
best_aic = float("inf")
best_order = None
best_seasonal_order = None
best_model = None

# Grid search over all parameter combinations
for order in order_combinations:
    for seasonal_order in seasonal_combinations:
        try:
            #TO DO: Train SARIMA model
            model = SARIMAX(
                train_target,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            model_fit = model.fit(disp=False)
            #TO DO: Check AIC
            if model_fit.aic < best_aic:
                best_aic = model_fit.aic
                best_order = order
                best_seasonal_order = seasonal_order
                best_model = model_fit

        except Exception as e:
            print(e)
            continue

print(f"Best SARIMA Model: Order={best_order}, Seasonal_Order={best_seasonal_order}, AIC={best_aic}")
print(best_model.summary())

In [ ]:
from tqdm import tqdm
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd

history = train_target.squeeze().astype(float).tolist()

sarima_predictions = []

for i in tqdm(range(len(test_data)), desc="SARIMA Rolling Forecast"):

    model = SARIMAX(
        history,
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    model_fit = model.fit(disp=False)

    # Predict one step ahead
    pred = model_fit.forecast(steps=1)

    sarima_predictions.append(float(pred[0]))

    # Add the REAL observation
    history.append(float(test_data.iloc[i, 0]))

sarima_predictions = pd.Series(
    sarima_predictions,
    index=test_data.index
)

In [ ]:
sarima_predictions_np = sarima_predictions.to_numpy()
sarima_predictions_np.shape

In [ ]:
test_real = test_data["Close"].to_numpy()
test_real.shape

In [ ]:
plot_predictions(
    sarima_predictions_np,
    test_real,
    "SARIMA"
)

In [ ]:
import joblib

# ===============================
# Save ARIMA
# ===============================
joblib.dump(
    arima_model,
    (Results_root +  "ARIMA.pkl")
)

# ===============================
# Save SARIMA
# ===============================
joblib.dump(
    best_model,
        (Results_root +  "SARIMA.pkl")
)

print("✅ ARIMA and SARIMA models saved successfully.")